# De la exploración al reporte de mediciones

La primera notebook nos permitió comprobar formas, seleccionar rondas y resumir
columnas. Ahora usaremos esas operaciones como funciones reutilizables desde la
notebook y desde `main.py`, sin reconstruir la lógica en cada celda.

## Contenido

1. Cargar y validar la forma.
2. Seleccionar rondas completas.
3. Resumir por sala e interpretar el resultado.
4. Añadir una selección propia.

Abre esta notebook en VS Code desde `02-reporte-mediciones` y selecciona el kernel de `.venv`.


In [1]:
import json
from pathlib import Path

import numpy as np

from measurements.processing import (
    ROOM_NAMES,
    load_readings,
    select_complete_rows,
    summarize,
)

project_dir = Path.cwd()
data_dir = project_dir.parent / "datos"
if not (project_dir / "main.py").is_file() or not data_dir.is_dir():
    raise FileNotFoundError(
        "Start the notebook from the 02-reporte-mediciones project directory"
    )

## 1. Cargar y validar la forma

`load_readings` comprueba el encabezado y lee una matriz con tres columnas.
La comprobación impide asignar un resumen a otra sala si cambia el orden del CSV.
Abre el módulo en el editor e identifica qué parte lee y cuál valida.

[Lectura con loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [2]:
readings = load_readings(data_dir / "readings.csv")
print(ROOM_NAMES)
print(readings.shape)
print(readings[:2])
assert readings.shape == (8, 3)

('north_room', 'south_room', 'storage_room')
(8, 3)
[[22. 24. 20.]
 [23. 25. 21.]]


## 2. Seleccionar rondas completas

La función devuelve dos resultados: las filas seleccionadas y la cantidad de
rechazos. Esto permite conservar evidencia del filtrado. Ni el arreglo original
ni el CSV se modifican.

In [3]:
quality_readings = load_readings(data_dir / "readings_quality.csv")
original = quality_readings.copy()
complete, rejected = select_complete_rows(quality_readings)
print(complete)
print(f"Rejected rows: {rejected}")
assert rejected == 2
assert not np.shares_memory(complete, quality_readings)
np.testing.assert_array_equal(quality_readings, original)

[[22. 24. 20.]
 [24. 26. 22.]]
Rejected rows: 2


### Comprobación: ninguna ronda completa

No calcularemos medias sobre una matriz vacía. La función comunica el problema
con una excepción; la notebook puede capturarla para discutir el caso y continuar.

In [4]:
invalid_readings = np.array([[np.nan, 24, 20], [22, np.inf, 21]])
try:
    select_complete_rows(invalid_readings)
except ValueError as error:
    print(f"Expected error: {error}")
else:
    raise AssertionError("Expected an error for data without complete rows")

Expected error: No complete readings remain


## 3. Resumir por sala

Las operaciones numéricas se aplican por eje. El pequeño ciclo de la función
`summarize` asigna nombres a los tres resultados y convierte los escalares a
`float` de Python para serializarlos en JSON. No recorre las mediciones para
calcular sus medias.

In [5]:
report = summarize(complete)
print(json.dumps(report, indent=2, allow_nan=False))
np.testing.assert_allclose(
    [report[name]["mean_c"] for name in ROOM_NAMES], [23, 25, 21]
)

{
  "north_room": {
    "mean_c": 23.0,
    "min_c": 22.0,
    "max_c": 24.0
  },
  "south_room": {
    "mean_c": 25.0,
    "min_c": 24.0,
    "max_c": 26.0
  },
  "storage_room": {
    "mean_c": 21.0,
    "min_c": 20.0,
    "max_c": 22.0
  }
}


### Ejercicio 10: explicar el cambio de medias

Compara el reporte de las ocho rondas originales con el de las dos rondas
completas del archivo con errores. Describe qué datos entraron en cada cálculo.
No son dos momentos de una instalación real: son archivos de práctica distintos.

In [6]:
# Compare the summaries and write your interpretation.
full_readings = load_readings(data_dir / "readings.csv")
full_report = summarize(full_readings)  # las 8 rondas
partial_report = summarize(
    complete
)  # solo las 2 rondas completas del archivo con errores

for name in ROOM_NAMES:
    print(name, full_report[name]["mean_c"], partial_report[name]["mean_c"])

np.testing.assert_allclose(
    [full_report[name]["mean_c"] for name in ROOM_NAMES], [25.5, 27.5, 23.5]
)
np.testing.assert_allclose(
    [partial_report[name]["mean_c"] for name in ROOM_NAMES], [23, 25, 21]
)

# REVISAR: reescribe con tus palabras.
# Reporte completo: entraron las ocho rondas de readings.csv (de 22 a 29 en la sala
# norte); las medias son 25.5, 27.5 y 23.5.
# Reporte parcial: de readings_quality.csv solo entraron las rondas 0 y 2
# (22, 24, 20 y 24, 26, 22), porque las otras dos tenian nan e inf y se descartaron
# completas; las medias son 23, 25 y 21.
# Las medias cambian porque los datos son otros: son archivos de practica distintos y
# no dos momentos de una misma instalacion, asi que no se puede decir que las salas
# se enfriaron.

north_room 25.5 23.0
south_room 27.5 25.0
storage_room 23.5 21.0


## 4. Añadir una selección propia

### Ejercicio 11: seleccionar rondas cálidas

Resuelve la [práctica](../PRACTICA.md) creando `select_warm_rounds` en un módulo
nuevo. Reinicia el kernel después de guardar el módulo y comprueba aquí las filas
seleccionadas mediante `assert`.

Para umbral `27`, deben quedar tres rondas del archivo original. Para umbral
`100`, la selección es vacía y no se debe calcular su media.

In [7]:
# Import your selection function and verify the results here.
from measurements.selection import select_warm_rounds

warm = select_warm_rounds(readings, 27)
np.testing.assert_array_equal(warm, [[27, 29, 25], [28, 30, 26], [29, 31, 27]])
assert warm.shape == (3, 3)

# Umbral exacto: la ronda con norte igual a 27 si entra (>=).
assert select_warm_rounds(readings, 29).shape == (1, 3)

# Ninguna coincidencia: forma (0, 3); no se calcula su media.
none_selected = select_warm_rounds(readings, 100)
assert none_selected.shape == (0, 3)

# Valores no finitos: datos o umbral invalidos producen ValueError.
for bad_readings, bad_threshold in [
    (np.array([[np.nan, 24, 20]]), 27),
    (np.array([[22, np.inf, 20]]), 27),
    (readings, np.nan),
    (readings, np.inf),
]:
    try:
        select_warm_rounds(bad_readings, bad_threshold)
    except ValueError as error:
        print(f"Expected error: {error}")
    else:
        raise AssertionError("Expected ValueError")

# Una seleccion modificable no cambia el original.
original_readings = readings.copy()
warm[0, 0] = -1
np.testing.assert_array_equal(readings, original_readings)
assert not np.shares_memory(readings, warm)

# Comparacion de medias.
warm = select_warm_rounds(readings, 27)
print("Seleccion:", warm.mean(axis=0))
print("Completo: ", readings.mean(axis=0))
np.testing.assert_allclose(warm.mean(axis=0), [28, 30, 26])
np.testing.assert_allclose(readings.mean(axis=0), [25.5, 27.5, 23.5])

# REVISAR: reescribe con tus palabras.
# Con umbral 100 la seleccion tiene forma (0, 3): no hay ninguna fila, asi que no se
# calcula la media (daria nan con una advertencia); se debe detectar que esta vacia.
# En la media de la seleccion entraron solo las tres rondas con norte >= 27
# (filas 5, 6 y 7); en la media completa entraron las ocho rondas. Por eso la media
# de la seleccion es mayor: [28, 30, 26] frente a [25.5, 27.5, 23.5].

Expected error: Readings must be finite
Expected error: Readings must be finite
Expected error: Threshold must be finite
Expected error: Threshold must be finite
Seleccion: [28. 30. 26.]
Completo:  [25.5 27.5 23.5]


## Verificación final

Desde la terminal del proyecto ejecuta `uv run --locked python main.py` para
generar los archivos. Después reinicia el kernel y ejecuta toda la notebook.
Las pruebas existentes con pytest son una comprobación opcional.
